In [1]:
from pathlib import Path
import json
import math
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats
import seaborn as sns
from BorutaShap import BorutaShap
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from xgboost import XGBClassifier

MODEL_DIR = "xgboost"
MODEL_NAME = "XGBoost"

cwd = Path.cwd()
project_root = Path("..").resolve() if cwd.name == MODEL_DIR else cwd
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from qsar_config import DATA_PATH, RANDOM_SEED
from qsar_common import (
    add_mol_column,
    aggregate_targets_by_fingerprint,
    build_morgan_fingerprints,
    encode_targets,
    load_qsar_dataset,
    stack_features_and_targets,
)

if not hasattr(np, "float"):
    np.float = float
if not hasattr(np, "int"):
    np.int = int

if not hasattr(scipy.stats, "binom_test"):
    def binom_test_wrapper(x, n=None, p=0.5, alternative="two-sided"):
        k_clean = int(x)
        n_clean = int(n) if n is not None else None
        return scipy.stats.binomtest(k=k_clean, n=n_clean, p=p, alternative=alternative).pvalue
    scipy.stats.binom_test = binom_test_wrapper

In [2]:
BORUTA_TRIALS = 20
FPSIZE = 4096

In [3]:
df = load_qsar_dataset(DATA_PATH)

In [4]:
df = add_mol_column(df, smiles_column="Smiles", mol_column="mol")

In [5]:
df = build_morgan_fingerprints(
    df,
    mol_column="mol",
    output_column="fp",
    radius=2,
    n_bits=FPSIZE,
)

In [6]:
df, encoder, target_names = encode_targets(
    df,
    target_column="Target",
    output_column="target_encoded",
)

In [7]:
df_agg = aggregate_targets_by_fingerprint(
    df,
    fp_column="fp",
    encoded_target_column="target_encoded",
    aggregated_target_column="target",
)

In [8]:
x, y = stack_features_and_targets(
    df_agg,
    fp_column="fp",
    target_column="target",
)

In [9]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.3,
    random_state=RANDOM_SEED,
)

In [10]:
import numpy as np
import scipy.stats

if not hasattr(np, 'NaN'):
    np.NaN = np.nan
if not hasattr(np, 'float'):
    np.float = float
if not hasattr(np, 'int'):
    np.int = int

if not hasattr(scipy.stats, 'binom_test'):
    def binom_test_wrapper(x, n=None, p=0.5, alternative='two-sided'):
        k_clean = int(x)
        n_clean = int(n) if n is not None else None

        return scipy.stats.binomtest(k=k_clean, n=n_clean, p=p, alternative=alternative).pvalue

    scipy.stats.binom_test = binom_test_wrapper

from BorutaShap import BorutaShap
import pandas as pd

feature_names = [f"Bit_{i}" for i in range(x.shape[1])]
X_df = pd.DataFrame(x, columns=feature_names)

target_names = encoder.categories_[0]

print(f"Feature Matrix Shape: {X_df.shape}")
print(f"Targets to analyze: {target_names}")

Feature Matrix Shape: (7207, 4096)
Targets to analyze: ['ar' 'era' 'erb' 'gr' 'mr' 'pr']


In [11]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from BorutaShap import BorutaShap
from xgboost import XGBClassifier

repo_root = Path.cwd().resolve()
if repo_root.name == "xgboost":
    repo_root = repo_root.parent.parent
elif repo_root.name == "qsar":
    repo_root = repo_root.parent

qsar_dir = repo_root / "qsar"
if str(qsar_dir) not in sys.path:
    sys.path.insert(0, str(qsar_dir))

from checkpoint_utils import file_signature, load_pickle_cache, save_pickle_cache

CACHE_VERSION = 1
cache_dir = repo_root / "qsar" / "xgboost" / "checkpoints" / "boruta-shap"
cache_dir.mkdir(parents=True, exist_ok=True)
plot_dir = Path("images") / "boruta-shap" / "xgboost"
plot_dir.mkdir(parents=True, exist_ok=True)

feature_names = [f"bit_{i}" for i in range(x.shape[1])]
X_boruta = pd.DataFrame(x, columns=feature_names)
selected_features_per_target = {}

xgb_boruta = XGBClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=RANDOM_SEED,
    tree_method="hist",
    eval_metric="logloss",
)

data_path = (qsar_dir / "nr_ic_merged.csv").resolve()
data_sig = file_signature(data_path)

In [12]:
zscore_results = []
PER_TARGET_TOP_N = 20

for i, target_name in enumerate(target_names):
    print(f"\n--- Processing target: {target_name} ---")
    y_target = pd.Series(y[:, i], name=str(target_name))

    safe_name = str(target_name).replace("/", "_").replace(" ", "_")
    boruta_cache_path = cache_dir / f"target_{safe_name}_scores.pkl"
    boruta_cache_meta = {
        "version": CACHE_VERSION,
        "kind": "qsar_boruta_target_scores",
        "target_name": str(target_name),
        "data": data_sig,
        "n_bits": int(x.shape[1]),
        "random_seed": int(RANDOM_SEED),
        "boruta_n_trials": int(BORUTA_TRIALS),
        "model": {
            "type": "XGBClassifier",
            "n_estimators": int(getattr(xgb_boruta, "n_estimators", 0)),
            "max_depth": int(getattr(xgb_boruta, "max_depth", 0) or 0),
            "learning_rate": float(getattr(xgb_boruta, "learning_rate", 0.0) or 0.0),
            "subsample": float(getattr(xgb_boruta, "subsample", 0.0) or 0.0),
            "colsample_bytree": float(getattr(xgb_boruta, "colsample_bytree", 0.0) or 0.0),
        },
    }

    cached = load_pickle_cache(boruta_cache_path, boruta_cache_meta)

    if cached is None:
        selector = BorutaShap(
            model=xgb_boruta,
            importance_measure="shap",
            classification=True,
        )

        try:
            selector.fit(
                X=X_boruta,
                y=y_target,
                n_trials=BORUTA_TRIALS,
                sample=False,
                verbose=True,
            )

            mean_z_scores = selector.history_x.mean(axis=0)
            mean_z_scores.name = str(target_name)

            selector.TentativeRoughFix()
            subset = selector.Subset()
            selected_features = subset.columns.tolist()

            save_pickle_cache(
                boruta_cache_path,
                boruta_cache_meta,
                {
                    "mean_z_scores": mean_z_scores,
                    "selected_features": selected_features,
                    "accepted": [str(v) for v in getattr(selector, "accepted", [])],
                    "tentative": [str(v) for v in getattr(selector, "tentative", [])],
                    "rejected": [str(v) for v in getattr(selector, "rejected", [])],
                },
            )
            cache_status = "miss"
        except Exception as e:
            print(f"    -> Failed for {target_name}: {e}")
            continue
    else:
        mean_z_scores = cached["mean_z_scores"]
        selected_features = cached.get("selected_features", [])
        cache_status = "hit"

    zscore_results.append(mean_z_scores)
    selected_features_per_target[target_name] = list(selected_features)

    target_abs = mean_z_scores.abs().sort_values(ascending=False)
    target_top = target_abs.head(PER_TARGET_TOP_N)
    if len(target_top) > 0:
        plt.figure(figsize=(10, max(4, 0.35 * len(target_top))))
        plt.barh(range(len(target_top)), target_top.values[::-1])
        plt.yticks(range(len(target_top)), target_top.index[::-1])
        plt.xlabel("Mean |Z-score|")
        plt.title(f"BorutaShap top bits: {target_name}")
        plt.tight_layout()
        target_plot_path = plot_dir / f"boruta_plot_{safe_name}.svg"
        plt.savefig(target_plot_path, format="svg")
        plt.close()

    print(f"    -> cache={cache_status} | selected features={len(selected_features)}")

final_zscore_df = pd.DataFrame(zscore_results).fillna(0)
print("Z-score matrix prepared in memory.")


--- Processing target: ar ---


  0%|          | 0/20 [00:00<?, ?it/s]

72 attributes confirmed important: ['bit_846', 'bit_3989', 'bit_194', 'bit_2793', 'bit_2857', 'bit_915', 'bit_3824', 'bit_4090', 'bit_1980', 'bit_2722', 'bit_960', 'bit_3259', 'bit_3839', 'bit_1721', 'bit_1535', 'bit_3996', 'bit_785', 'bit_350', 'bit_1236', 'bit_361', 'bit_807', 'bit_1686', 'bit_2322', 'bit_2944', 'bit_622', 'bit_3511', 'bit_262', 'bit_3738', 'bit_980', 'bit_1337', 'bit_371', 'bit_2855', 'bit_2817', 'bit_1696', 'bit_289', 'bit_3798', 'bit_1697', 'bit_2128', 'bit_1313', 'bit_1309', 'bit_3449', 'bit_2455', 'bit_39', 'bit_1845', 'bit_3900', 'bit_621', 'bit_3184', 'bit_5', 'bit_2049', 'bit_2572', 'bit_3814', 'bit_1536', 'bit_2438', 'bit_2988', 'bit_2929', 'bit_3524', 'bit_1162', 'bit_1862', 'bit_2536', 'bit_334', 'bit_2437', 'bit_899', 'bit_3797', 'bit_1959', 'bit_1601', 'bit_2467', 'bit_2838', 'bit_1115', 'bit_3986', 'bit_1160', 'bit_3200', 'bit_2067']
3976 attributes confirmed unimportant: ['bit_2050', 'bit_1291', 'bit_4061', 'bit_1968', 'bit_2449', 'bit_688', 'bit_2861'

  0%|          | 0/20 [00:00<?, ?it/s]

76 attributes confirmed important: ['bit_2799', 'bit_3475', 'bit_194', 'bit_2793', 'bit_3650', 'bit_4030', 'bit_2091', 'bit_2362', 'bit_2722', 'bit_2436', 'bit_3839', 'bit_2766', 'bit_841', 'bit_1009', 'bit_1270', 'bit_350', 'bit_4039', 'bit_361', 'bit_1686', 'bit_1475', 'bit_102', 'bit_2787', 'bit_3629', 'bit_262', 'bit_1357', 'bit_1239', 'bit_2727', 'bit_3744', 'bit_2429', 'bit_3959', 'bit_1928', 'bit_1233', 'bit_119', 'bit_1696', 'bit_3956', 'bit_3781', 'bit_1308', 'bit_3108', 'bit_1164', 'bit_1274', 'bit_2128', 'bit_1313', 'bit_2742', 'bit_41', 'bit_1582', 'bit_3578', 'bit_2983', 'bit_322', 'bit_2033', 'bit_1052', 'bit_3184', 'bit_3361', 'bit_1453', 'bit_1855', 'bit_1847', 'bit_3655', 'bit_1088', 'bit_1145', 'bit_3580', 'bit_2004', 'bit_1939', 'bit_2974', 'bit_1256', 'bit_3554', 'bit_2698', 'bit_13', 'bit_105', 'bit_4036', 'bit_1941', 'bit_2132', 'bit_2467', 'bit_162', 'bit_1951', 'bit_1160', 'bit_3200', 'bit_1670']
3944 attributes confirmed unimportant: ['bit_2050', 'bit_1291', 'b

...

68 attributes confirmed important: ['bit_1687', 'bit_3650', 'bit_2604', 'bit_2793', 'bit_2857', 'bit_695', 'bit_1970', 'bit_2107', 'bit_2091', 'bit_2362', 'bit_3839', 'bit_841', 'bit_1553', 'bit_1535', 'bit_350', 'bit_333', 'bit_361', 'bit_2944', 'bit_807', 'bit_3770', 'bit_3087', 'bit_1215', 'bit_3082', 'bit_262', 'bit_2727', 'bit_3959', 'bit_3211', 'bit_3612', 'bit_371', 'bit_1928', 'bit_2457', 'bit_2932', 'bit_3956', 'bit_1237', 'bit_1274', 'bit_3067', 'bit_3108', 'bit_1313', 'bit_1436', 'bit_2742', 'bit_1147', 'bit_4006', 'bit_280', 'bit_1344', 'bit_2983', 'bit_2230', 'bit_3652', 'bit_2400', 'bit_1747', 'bit_1154', 'bit_3105', 'bit_389', 'bit_1145', 'bit_3786', 'bit_3349', 'bit_3458', 'bit_630', 'bit_13', 'bit_1279', 'bit_1941', 'bit_2132', 'bit_1257', 'bit_1421', 'bit_1536', 'bit_2854', 'bit_1160', 'bit_3200', 'bit_1670']
3965 attributes confirmed unimportant: ['bit_2050', 'bit_3989', 'bit_1291', 'bit_4061', 'bit_1968', 'bit_2449', 'bit_688', 'bit_2861', 'bit_3077', 'bit_3585', 'b

  0%|          | 0/20 [00:00<?, ?it/s]

29 attributes confirmed important: ['bit_3650', 'bit_1338', 'bit_2793', 'bit_3782', 'bit_441', 'bit_2107', 'bit_971', 'bit_841', 'bit_2766', 'bit_1101', 'bit_350', 'bit_361', 'bit_1594', 'bit_2518', 'bit_262', 'bit_2303', 'bit_1021', 'bit_1905', 'bit_692', 'bit_3219', 'bit_4006', 'bit_2049', 'bit_1060', 'bit_2929', 'bit_1145', 'bit_2974', 'bit_630', 'bit_132', 'bit_1670']
4025 attributes confirmed unimportant: ['bit_2050', 'bit_3989', 'bit_1291', 'bit_4061', 'bit_1968', 'bit_2449', 'bit_688', 'bit_2861', 'bit_3077', 'bit_3585', 'bit_552', 'bit_38', 'bit_1985', 'bit_1623', 'bit_3996', 'bit_1783', 'bit_3428', 'bit_28', 'bit_1161', 'bit_263', 'bit_247', 'bit_2021', 'bit_266', 'bit_3314', 'bit_1078', 'bit_1850', 'bit_2559', 'bit_3883', 'bit_3946', 'bit_2575', 'bit_1837', 'bit_1913', 'bit_384', 'bit_2147', 'bit_3740', 'bit_407', 'bit_1543', 'bit_3451', 'bit_1504', 'bit_1628', 'bit_2312', 'bit_3186', 'bit_1702', 'bit_4054', 'bit_2375', 'bit_1500', 'bit_457', 'bit_775', 'bit_3068', 'bit_416',

  0%|          | 0/20 [00:00<?, ?it/s]

59 attributes confirmed important: ['bit_3650', 'bit_2793', 'bit_2857', 'bit_1917', 'bit_695', 'bit_4030', 'bit_2281', 'bit_1980', 'bit_35', 'bit_960', 'bit_2413', 'bit_2722', 'bit_3839', 'bit_2522', 'bit_2730', 'bit_785', 'bit_361', 'bit_2944', 'bit_1475', 'bit_807', 'bit_1594', 'bit_262', 'bit_2682', 'bit_3500', 'bit_191', 'bit_3873', 'bit_3733', 'bit_2855', 'bit_372', 'bit_119', 'bit_3108', 'bit_1274', 'bit_1313', 'bit_2290', 'bit_3875', 'bit_3184', 'bit_991', 'bit_378', 'bit_3105', 'bit_746', 'bit_3330', 'bit_3028', 'bit_3655', 'bit_4041', 'bit_1145', 'bit_1957', 'bit_2698', 'bit_1638', 'bit_630', 'bit_127', 'bit_13', 'bit_216', 'bit_3802', 'bit_34', 'bit_439', 'bit_1257', 'bit_45', 'bit_1160', 'bit_253']
3979 attributes confirmed unimportant: ['bit_2050', 'bit_3989', 'bit_1291', 'bit_4061', 'bit_1968', 'bit_2449', 'bit_688', 'bit_2861', 'bit_3077', 'bit_3585', 'bit_552', 'bit_38', 'bit_1985', 'bit_1623', 'bit_3996', 'bit_1783', 'bit_3428', 'bit_28', 'bit_1161', 'bit_263', 'bit_247

In [13]:
TOP_N = 20
z_abs = final_zscore_df.abs()
mean_abs = z_abs.mean(axis=0).sort_values(ascending=False)

if len(mean_abs) == 0:
    raise RuntimeError("Nebyla nalezena žádná Z-skóre pro vykreslení.")

top_bits = mean_abs.head(TOP_N).index.tolist()
plot_df = final_zscore_df[top_bits]

plt.figure(figsize=(max(10, TOP_N * 0.45), max(6, 0.35 * len(plot_df))))
row_order = plot_df.abs().sum(axis=1).sort_values(ascending=False).index
plot_df_sorted = plot_df.loc[row_order]

im = plt.imshow(plot_df_sorted.values, aspect="auto", cmap="coolwarm")
plt.colorbar(im, label="Průměrné Z-skóre (BorutaShap)")
plt.yticks(np.arange(plot_df_sorted.shape[0]), plot_df_sorted.index)
plt.xticks(np.arange(len(top_bits)), top_bits, rotation=90)
plt.title(f"Nejlepších {TOP_N} bitů podle průměru |Z| napříč cíli (BorutaShap)")
plt.tight_layout()
heatmap_path = plot_dir / f"boruta_top_{TOP_N}_zscore_heatmap.svg"
plt.savefig(heatmap_path, format="svg")
plt.close()

plt.figure(figsize=(max(10, TOP_N * 0.5), 5))
plt.bar(range(len(top_bits)), mean_abs.loc[top_bits].values)
plt.xticks(range(len(top_bits)), top_bits, rotation=90)
plt.ylabel("Průměr(|Z|) napříč cíli")
plt.title(f"Nejlepších {TOP_N} bitů podle průměru |Z| (BorutaShap)")
plt.tight_layout()
bar_path = plot_dir / f"boruta_top_{TOP_N}_zscore_bar.svg"
plt.savefig(bar_path, format="svg")
plt.close()

print(f"Saved top-N plots: {heatmap_path.name}, {bar_path.name}")

Saved top-N plots: boruta_top_20_zscore_heatmap.svg, boruta_top_20_zscore_bar.svg
